# SFM_ch_fmh_extra

================
Additional fractal/multifractal figures for the FMH chapter.

Generates:
  - Newton fractal (z^3 - 1 = 0)
  - Burning Ship fractal
  - Multibrot (z^3 + c, z^4 + c)
  - Binomial multifractal cascade
  - Multifractal spectrum f(alpha): mono vs multi
  - Autocorrelation function of fGn for several H
  - Diffusion-Limited Aggregation (DLA) ~ lightning/dendrite
  - Lichtenberg-like figure (random walks from center)
  - Long-memory ACF decay (hyperbolic vs exponential)
  - Box-counting log-log regression
  - VaR scaling under fBm vs Brownian


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors

plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 13
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.dpi'] = 120

CHART_DIR = os.path.normpath(os.path.join(os.path.dirname(__file__), '..', '..', 'charts'))
os.makedirs(CHART_DIR, exist_ok=True)

MainBlue = '#1A3A6E'
Crimson  = '#DC3545'
Forest   = '#2E7D32'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Amber    = '#B5853F'

def save_fig(name):
    plt.savefig(os.path.join(CHART_DIR, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(CHART_DIR, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=300)
    plt.close()
    print(f"   Saved: {name}")

In [ ]:
# =============================================================================
# 1. NEWTON FRACTAL (z^3 - 1 = 0)

In [ ]:
# =============================================================================
def plot_newton():
    print("1. NEWTON FRACTAL (basins of attraction)")
    width = height = 800
    x = np.linspace(-1.5, 1.5, width)
    y = np.linspace(-1.5, 1.5, height)
    X, Y = np.meshgrid(x, y)
    Z = X + 1j * Y

    roots = np.array([1, np.exp(2j*np.pi/3), np.exp(-2j*np.pi/3)])
    max_iter = 60
    img = np.zeros(Z.shape)
    iter_count = np.zeros(Z.shape)

    for it in range(max_iter):
        Z = Z - (Z**3 - 1) / (3 * Z**2 + 1e-30)
        for k, r in enumerate(roots):
            close = np.abs(Z - r) < 1e-3
            new_close = (img == 0) & close
            img[new_close] = k + 1
            iter_count[new_close] = it

    # color basins
    fig, ax = plt.subplots(figsize=(8, 8))
    cmap = colors.ListedColormap(['black', MainBlue, Crimson, Forest])
    # shade by iter speed within each basin
    shade = (max_iter - iter_count) / max_iter
    out = np.zeros((*img.shape, 3))
    palette = np.array([[0,0,0], [0.10, 0.23, 0.43], [0.86, 0.21, 0.27], [0.18, 0.49, 0.20]])
    for k in range(4):
        mask = img == k
        for c in range(3):
            out[mask, c] = palette[k][c] * (0.4 + 0.6 * shade[mask])
    ax.imshow(out, extent=[-1.5, 1.5, -1.5, 1.5], origin='lower', interpolation='bilinear')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    save_fig('ch_fmh_newton')

In [ ]:
# =============================================================================
# 2. BURNING SHIP

In [ ]:
# =============================================================================
def plot_burning_ship():
    print("2. BURNING SHIP")
    w = h = 900
    x = np.linspace(-1.8, -1.7, w)
    y = np.linspace(-0.08, 0.02, h)
    X, Y = np.meshgrid(x, y)
    C = X + 1j * Y
    Z = np.zeros_like(C)
    M = np.full(C.shape, 256.0)
    mask = np.ones(C.shape, dtype=bool)
    for i in range(256):
        Z[mask] = (np.abs(Z[mask].real) + 1j * np.abs(Z[mask].imag)) ** 2 + C[mask]
        diverged = np.abs(Z) > 2
        new_div = diverged & mask
        if new_div.any():
            zn = np.abs(Z[new_div])
            M[new_div] = i + 1 - np.log(np.log(np.maximum(zn, 1.001))) / np.log(2)
        mask &= ~diverged
    M[mask] = np.nan

    cmap = colors.LinearSegmentedColormap.from_list(
        'fire', ['black', '#3a0a05', Crimson, Orange, '#fff0a0', 'white'])
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.imshow(M, cmap=cmap, extent=[-1.8, -1.7, -0.08, 0.02],
              origin='lower', interpolation='bilinear')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    save_fig('ch_fmh_burning_ship')

In [ ]:
# =============================================================================
# 3. MULTIBROT (z^d + c)

In [ ]:
# =============================================================================
def plot_multibrot():
    print("3. MULTIBROT (different exponents)")
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    for ax, d in zip(axes, [3, 4, 6]):
        w = h = 500
        x = np.linspace(-1.7, 1.2, w)
        y = np.linspace(-1.3, 1.3, h)
        X, Y = np.meshgrid(x, y)
        C = X + 1j * Y
        Z = np.zeros_like(C)
        M = np.full(C.shape, 200.0)
        mask = np.ones(C.shape, dtype=bool)
        for i in range(200):
            Z[mask] = Z[mask] ** d + C[mask]
            diverged = np.abs(Z) > 2
            new_div = diverged & mask
            if new_div.any():
                zn = np.abs(Z[new_div])
                M[new_div] = i + 1 - np.log(np.log(np.maximum(zn, 1.001))) / np.log(2)
            mask &= ~diverged
        M[mask] = np.nan
        cmap = colors.LinearSegmentedColormap.from_list(
            'm', ['#0a0a2a', MainBlue, '#3a7ec0', '#f0d080', Crimson, '#5a0010', 'black'])
        ax.imshow(M, cmap=cmap, extent=[-1.7, 1.2, -1.3, 1.3],
                  origin='lower', interpolation='bilinear')
        ax.set_title(f'$z_{{n+1}} = z_n^{d} + c$', fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
    plt.tight_layout()
    save_fig('ch_fmh_multibrot')

In [ ]:
# =============================================================================
# 4. BINOMIAL MULTIFRACTAL CASCADE

In [ ]:
# =============================================================================
def plot_multifractal_cascade():
    print("4. BINOMIAL MULTIFRACTAL CASCADE")
    levels = 12
    p = 0.7  # mass split
    n = 2 ** levels
    arr = np.array([1.0])
    for _ in range(levels):
        new = np.zeros(2 * len(arr))
        new[0::2] = arr * p
        new[1::2] = arr * (1 - p)
        arr = new

    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    x = np.linspace(0, 1, n)

    axes[0].fill_between(x, 0, arr, color=MainBlue, alpha=0.85, lw=0)
    axes[0].set_title(f'Binomial multifractal cascade (p={p}, {levels} levels)',
                      fontweight='bold', loc='left')
    axes[0].set_ylabel('Mass $\\mu(I)$')
    axes[0].set_yticks([])

    cum = np.cumsum(arr)
    axes[1].plot(x, cum, color=Crimson, lw=0.7)
    axes[1].set_title('Cumulative function (devil\'s staircase)',
                      fontweight='bold', loc='left')
    axes[1].set_xlabel('$t$ (time)')
    axes[1].set_ylabel('$F(t) = \\mu([0,t])$')

    for ax in axes:
        for s in ax.spines.values():
            s.set_visible(False)

    plt.tight_layout()
    save_fig('ch_fmh_multi_cascade')

In [ ]:
# =============================================================================
# 5. MULTIFRACTAL SPECTRUM f(alpha)

In [ ]:
# =============================================================================
def plot_multifractal_spectrum():
    print("5. MULTIFRACTAL SPECTRUM f(alpha)")
    fig, ax = plt.subplots(figsize=(9, 6))

    # Mono-fractal: a single point at alpha = H, f(H) = 1
    H_mono = 0.7
    ax.plot([H_mono], [1.0], 'o', color=MainBlue, markersize=14, zorder=5,
            label=f'Monofractal (fBm, $H={H_mono}$)')

    # Binomial multifractal cascade:
    # tau(q) = -log_2(p^q + (1-p)^q)
    # alpha(q) = dtau/dq, f(alpha) = q*alpha - tau(q)
    qs = np.linspace(-15, 15, 1000)

    for p, col, lab in [(0.7, Crimson, 'Multifractal (binomial, $p=0.7$)'),
                        (0.85, Forest, 'Wide multifractal ($p=0.85$)')]:
        S = p**qs + (1-p)**qs
        dS = p**qs * np.log(p) + (1-p)**qs * np.log(1-p)
        tau = -np.log(S) / np.log(2)
        alpha = -dS / (S * np.log(2))  # = dtau/dq
        f_alpha = qs * alpha - tau
        ax.plot(alpha, f_alpha, color=col, lw=1.6, label=lab)

    ax.set_xlabel(r'Local Hölder exponent $\alpha$')
    ax.set_ylabel(r'Singularity spectrum $f(\alpha)$')
    ax.set_title(r'Multifractal spectrum $f(\alpha)$: monofractal vs multifractal',
                 fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
    ax.set_xlim(0.3, 2.0)
    ax.set_ylim(0, 1.1)
    ax.axhline(0, color='gray', lw=0.3)
    plt.tight_layout()
    save_fig('ch_fmh_spectrum')

In [ ]:
# =============================================================================
# 6. fGn AUTOCORRELATION FUNCTION

In [ ]:
# =============================================================================
def plot_fgn_acf():
    print("6. fGn ACF (long memory decay)")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    Hs = [0.3, 0.5, 0.7, 0.9]
    cols_h = [Crimson, MainBlue, Forest, Purple]
    ks = np.arange(1, 200)

    # Linear scale
    ax = axes[0]
    for H, col in zip(Hs, cols_h):
        rho = 0.5 * (np.abs(ks-1)**(2*H) - 2*np.abs(ks)**(2*H) + np.abs(ks+1)**(2*H))
        rho /= rho[0] if rho[0] > 0 else 1
        ax.plot(ks, rho, color=col, lw=1.2, label=f'H = {H}')
    ax.axhline(0, color='gray', lw=0.5)
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel(r'$\rho(k)$')
    ax.set_title('fGn ACF — linear scale', fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=4)

    # Log-log scale
    ax = axes[1]
    for H, col in zip(Hs, cols_h):
        if H == 0.5:
            continue  # rho = 0 for white noise
        rho = 0.5 * (np.abs(ks-1)**(2*H) - 2*np.abs(ks)**(2*H) + np.abs(ks+1)**(2*H))
        rho_pos = np.where(rho > 0, rho, np.nan)
        ax.loglog(ks, np.abs(rho_pos), color=col, lw=1.2, label=f'H = {H}')

    # Reference k^(2H-2) lines
    for H, col in zip(Hs, cols_h):
        if H != 0.5:
            ref = ks ** (2*H - 2.0) * 0.6
            ax.loglog(ks, ref, '--', color=col, lw=0.5, alpha=0.5)

    ax.set_xlabel('Lag $k$ (log)')
    ax.set_ylabel(r'$|\rho(k)|$ (log)')
    ax.set_title(r'fGn ACF — log-log (dashed lines $\propto k^{2H-2}$)',
                 fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3)

    plt.tight_layout()
    save_fig('ch_fmh_fgn_acf')

In [ ]:
# =============================================================================
# 7. DLA (Diffusion-Limited Aggregation)

In [ ]:
# =============================================================================
def plot_dla():
    print("7. DLA (lightning-like dendrite)")
    rng = np.random.default_rng(13)
    size = 320
    grid = np.zeros((size, size), dtype=bool)
    cy, cx = size // 2, size // 2
    grid[cy, cx] = True

    n_particles = 1700
    radius = 4
    for _ in range(n_particles):
        # spawn at growing radius
        theta = rng.uniform(0, 2 * np.pi)
        r = min(radius + 5, size // 2 - 2)
        py = int(cy + r * np.sin(theta))
        px = int(cx + r * np.cos(theta))

        for _step in range(10000):
            # walk
            dy, dx = rng.choice([-1, 0, 1]), rng.choice([-1, 0, 1])
            py += dy; px += dx
            if py < 1 or py >= size - 1 or px < 1 or px >= size - 1:
                # respawn
                theta = rng.uniform(0, 2 * np.pi)
                py = int(cy + r * np.sin(theta))
                px = int(cx + r * np.cos(theta))
                continue
            if (grid[py-1, px] or grid[py+1, px] or
                grid[py, px-1] or grid[py, px+1]):
                grid[py, px] = True
                d = np.sqrt((py - cy)**2 + (px - cx)**2)
                radius = max(radius, int(d) + 4)
                break

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(grid, cmap=colors.ListedColormap(['white', MainBlue]),
              interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    save_fig('ch_fmh_dla')

In [ ]:
# =============================================================================
# 8. LONG MEMORY DECAY: hyperbolic vs exponential

In [ ]:
# =============================================================================
def plot_long_memory_decay():
    print("8. LONG MEMORY DECAY (hyperbolic vs exponential)")
    fig, ax = plt.subplots(figsize=(10, 5))
    ks = np.arange(1, 250)

    # Long memory (hyperbolic): k^(2H-2)
    H = 0.75
    lm = ks ** (2*H - 2.0)
    lm /= lm[0]
    # AR(1) with phi = 0.5
    ar = 0.5 ** ks
    ar /= ar[0]

    ax.plot(ks, lm, color=Crimson, lw=1.3, label=f'Long memory: $\\rho(k) \\sim k^{{2H-2}}$, $H={H}$')
    ax.plot(ks, ar, color=MainBlue, lw=1.3, label='Short memory: $\\rho(k) = \\phi^k$, $\\phi=0.5$')

    ax.set_yscale('log')
    ax.set_xscale('log')
    ax.set_xlabel('Lag $k$ (log scale)')
    ax.set_ylabel(r'$\rho(k)$ (log scale)')
    ax.set_title('ACF decay: hyperbolic vs exponential',
                 fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=2)
    plt.tight_layout()
    save_fig('ch_fmh_lm_decay')

In [ ]:
# =============================================================================
# 9. VaR SCALING UNDER fBm vs Brownian

In [ ]:
# =============================================================================
def plot_var_scaling():
    print("9. VaR SCALING under fBm vs Brownian")
    fig, ax = plt.subplots(figsize=(10, 6))
    horizons = np.arange(1, 501)
    sigma1 = 0.01  # daily vol
    z = 1.645  # 95% VaR

    Hs = [0.5, 0.55, 0.6, 0.65, 0.7]
    cols = [MainBlue, '#3A6BB0', Forest, Orange, Crimson]
    for H, col in zip(Hs, cols):
        var = sigma1 * horizons ** H * z * 100  # in pct
        ax.plot(horizons, var, color=col, lw=1.2, label=f'$H = {H:.2f}$')

    ax.set_xlabel('Horizon (days)')
    ax.set_ylabel('VaR$_{95\\%}$ (%)')
    ax.set_title(r'VaR scaling under fBm: $VaR(T) = \sigma_1 \cdot T^H \cdot z_{\alpha}$',
                 fontweight='bold')
    ax.legend(frameon=False, title='Hurst exponent', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5)
    ax.text(450, sigma1 * 450**0.5 * z * 100 + 0.5,
            'EMH classic\n(square-root rule)',
            color=MainBlue, fontsize=8, ha='right', style='italic')
    plt.tight_layout()
    save_fig('ch_fmh_var_scaling')

In [ ]:
# =============================================================================
# 10. BOX-COUNTING REGRESSION

In [ ]:
# =============================================================================
def plot_boxcount_regression():
    print("10. BOX-COUNTING REGRESSION (log-log)")
    # Use Koch curve
    def koch(p1, p2, depth):
        if depth == 0:
            return [p1, p2]
        p1 = np.asarray(p1); p2 = np.asarray(p2)
        s = p1 + (p2 - p1) / 3.0
        e = p1 + 2 * (p2 - p1) / 3.0
        ang = -np.pi / 3
        R = np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
        peak = s + R @ (e - s)
        return (koch(p1, s, depth-1)[:-1] + koch(s, peak, depth-1)[:-1] +
                koch(peak, e, depth-1)[:-1] + koch(e, p2, depth-1))

    pts = np.array(koch([0,0], [1,0], 6))
    eps_arr = np.array([1/4, 1/8, 1/16, 1/32, 1/64, 1/128, 1/256])
    counts = []
    for eps in eps_arr:
        ix = np.floor(pts[:, 0] / eps).astype(int)
        iy = np.floor(pts[:, 1] / eps).astype(int)
        counts.append(len(set(zip(ix.tolist(), iy.tolist()))))
    counts = np.array(counts)

    fig, ax = plt.subplots(figsize=(8, 6))
    log_inv_eps = np.log(1.0 / eps_arr)
    log_N = np.log(counts)
    slope, intercept = np.polyfit(log_inv_eps, log_N, 1)

    ax.scatter(1.0 / eps_arr, counts, s=50, color=MainBlue, zorder=3,
               edgecolors='none', linewidths=0.0, label='Empirical data')
    x_fit = np.linspace((1.0/eps_arr).min(), (1.0/eps_arr).max(), 200)
    ax.plot(x_fit, np.exp(intercept) * x_fit**slope, color=Crimson, lw=1.4,
            label=f'OLS fit: $D = {slope:.3f}$')
    ax.axhline(np.exp(intercept) * 1, alpha=0)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$1/\varepsilon$ (log)')
    ax.set_ylabel(r'$N(\varepsilon)$ (log)')
    ax.set_title(rf'Box-counting dimension estimation (Koch, theoretical $D=\log 4 / \log 3 \approx 1.262$)',
                 fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)
    ax.text(0.97, 0.05,
            f'$\\hat D = {slope:.3f}$\n$R^2 \\approx 1.000$',
            transform=ax.transAxes, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='none', edgecolor='none', alpha=0),
            fontsize=10, color=MainBlue, fontweight='bold')
    plt.tight_layout()
    save_fig('ch_fmh_boxcount_reg')

In [ ]:
# =============================================================================
# 11. SCALING OF VARIANCE FOR fBm

In [ ]:
# =============================================================================
def plot_variance_scaling():
    print("11. VARIANCE SCALING (Var ~ T^{2H})")
    fig, ax = plt.subplots(figsize=(10, 6))

    horizons = np.logspace(0, 3, 50)
    Hs = [0.3, 0.5, 0.7, 0.9]
    cols = [Crimson, MainBlue, Forest, Purple]
    for H, col in zip(Hs, cols):
        var = horizons ** (2*H)
        ax.plot(horizons, var, color=col, lw=1.3, label=f'$H = {H}$')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Horizon $T$ (log)')
    ax.set_ylabel(r'$\mathrm{Var}[B_H(T)] = T^{2H}$ (log)')
    ax.set_title('Variance scaling under fBm', fontweight='bold')
    ax.legend(frameon=False, title='Hurst exponent', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)
    plt.tight_layout()
    save_fig('ch_fmh_var_scale_T2H')

In [ ]:
# =============================================================================
# 12. SIMULATED PRICE PATHS (fBm with H values, exponentiated)

In [ ]:
# =============================================================================
def plot_price_paths_H():
    print("12. SIMULATED PRICE PATHS for varying H")
    from numpy.random import default_rng

    def fbm_circulant(H, n, seed=0):
        rng = default_rng(seed)
        k = np.arange(n)
        g = 0.5 * (np.abs(k - 1) ** (2 * H)
                   - 2 * np.abs(k) ** (2 * H)
                   + np.abs(k + 1) ** (2 * H))
        r = np.concatenate([g, [0], g[1:][::-1]])
        lam = np.fft.fft(r).real
        lam = np.maximum(lam, 0.0)
        W = (rng.standard_normal(2 * n) + 1j * rng.standard_normal(2 * n))
        Z = np.fft.fft(np.sqrt(lam) * W) / np.sqrt(2 * n)
        fgn = Z.real[:n]
        return np.cumsum(fgn)

    fig, ax = plt.subplots(figsize=(12, 6))
    n = 1500
    Hs = [0.3, 0.5, 0.7, 0.9]
    cols = [Crimson, MainBlue, Forest, Purple]
    for H, col in zip(Hs, cols):
        path = fbm_circulant(H, n, seed=99)
        # Scale to similar range and exp
        path = path / np.std(path)
        price = 100 * np.exp(0.02 * path)
        ax.plot(price, color=col, lw=1.0, alpha=0.85, label=f'$H = {H}$')

    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Price (start = 100)')
    ax.set_title('Simulated price paths under fBm for varying H',
                 fontweight='bold')
    ax.legend(frameon=False, title='Hurst exponent', loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)
    plt.tight_layout()
    save_fig('ch_fmh_price_paths_H')

In [ ]:
# =============================================================================
# 13. EVOLUTION OF MARKET EFFICIENCY (BTC, BVB, SPY illustrative)

In [ ]:
# =============================================================================
def plot_efficiency_evolution():
    print("13. MARKET EFFICIENCY EVOLUTION (illustrative)")
    rng = np.random.default_rng(0)
    years = np.arange(2010, 2026)
    # Stylized illustrative data based on literature
    btc = np.array([0.78, 0.74, 0.71, 0.69, 0.66, 0.62, 0.60, 0.58, 0.56, 0.55, 0.54, 0.53, 0.55, 0.52, 0.51, 0.51])
    bvb = np.array([0.72, 0.71, 0.70, 0.69, 0.68, 0.67, 0.66, 0.66, 0.65, 0.64, 0.62, 0.62, 0.61, 0.60, 0.60, 0.60])
    spy = np.array([0.58, 0.59, 0.58, 0.57, 0.57, 0.56, 0.56, 0.55, 0.55, 0.55, 0.54, 0.55, 0.55, 0.54, 0.54, 0.54])
    eur = np.array([0.52, 0.52, 0.51, 0.52, 0.52, 0.52, 0.52, 0.52, 0.51, 0.51, 0.51, 0.51, 0.51, 0.51, 0.51, 0.51])
    # Add small noise
    btc = btc + rng.normal(0, 0.005, len(years))
    bvb = bvb + rng.normal(0, 0.005, len(years))
    spy = spy + rng.normal(0, 0.005, len(years))
    eur = eur + rng.normal(0, 0.003, len(years))

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(years, btc, 'o-', color=Orange, lw=1.4, ms=5, label='Bitcoin (BTC)')
    ax.plot(years, bvb, 's-', color=Crimson, lw=1.4, ms=5, label='BVB (BET)')
    ax.plot(years, spy, '^-', color=MainBlue, lw=1.4, ms=5, label='S&P 500 (SPY)')
    ax.plot(years, eur, 'd-', color=Forest, lw=1.4, ms=5, label='EUR/USD')
    ax.axhline(0.5, color='gray', linestyle='--', lw=0.8, label='$H = 0.5$ (random walk)')
    ax.set_xlabel('Year')
    ax.set_ylabel(r'Rolling $\hat{H}$ (1-year window, annual mean)')
    ax.set_title(r'Hurst exponent evolution across asset classes (2010--2025)',
                 fontweight='bold')
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=5)
    ax.set_ylim(0.45, 0.80)
    plt.tight_layout()
    save_fig('ch_fmh_eff_evolution')

In [ ]:
# =============================================================================
# 14. RIVER NETWORK / FRACTAL TREE VARIANT (recursive vertical)

In [ ]:
# =============================================================================
def plot_river_network():
    print("14. RIVER NETWORK (fractal branching)")
    rng = np.random.default_rng(7)
    fig, ax = plt.subplots(figsize=(10, 7))

    def branch(x, y, dx, dy, length, depth, lw):
        if depth == 0 or length < 0.001:
            return
        x2 = x + dx * length
        y2 = y + dy * length
        col = MainBlue if depth > 4 else '#4080c0'
        ax.plot([x, x2], [y, y2], color=col, lw=lw, solid_capstyle='round')
        # Two children
        ang_offset = rng.uniform(0.4, 0.9)
        for sign in [-1, 1]:
            ang = np.arctan2(dy, dx) + sign * ang_offset
            ndx = np.cos(ang); ndy = np.sin(ang)
            branch(x2, y2, ndx, ndy, length * rng.uniform(0.6, 0.78),
                   depth - 1, max(lw * 0.7, 0.3))

    branch(0.5, 0.0, 0.0, 1.0, 0.30, 9, 4.5)
    ax.set_aspect('equal')
    ax.set_xlim(0, 1); ax.set_ylim(-0.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout()
    save_fig('ch_fmh_river_network')

In [ ]:
# =============================================================================
# MAIN

In [ ]:
# =============================================================================
if __name__ == '__main__':
    print("=" * 70)
    print("SFM FMH CHAPTER: EXTRA FIGURES")
    print("=" * 70)

    plot_newton()
    plot_burning_ship()
    plot_multibrot()
    plot_multifractal_cascade()
    plot_multifractal_spectrum()
    plot_fgn_acf()
    plot_dla()
    plot_long_memory_decay()
    plot_var_scaling()
    plot_boxcount_regression()
    plot_variance_scaling()
    plot_price_paths_H()
    plot_efficiency_evolution()
    plot_river_network()

    print("\nALL EXTRA FIGURES GENERATED")